# 2. 常用内置中间件的使用
LangChain 1.0 提供了 16 个预置中间件，开箱即用。

我们不会逐个介绍上述与模型供应商无关的所有内置中间件，这里将与模型供应商无关的内置中间件分成两部分： 常用内置中间件 （本节）， 其它内置中间件 （下节）。

第一部分将会详细讲解，第二部分提供测试代码和结果，快速演示。

## 2.1 SummarizationMiddleware中间件
* 作用：对历史消息列表进行 摘要&总结 ，达到 压缩上下文 的效果。
* 原理：在 达到触发条件 时，调用大模型对历史消息进行摘要， 将摘要的结果作为`HumanMessage` ，放到消息列表`最开始的位置`。

这个中间件会监控token数量，当达到某个阈值时，它会自动对旧消息进行汇总处理。这样既能保留最新的消息，又能保持上下文的连续性，确保人工智能/工具之间的消息配对关系始终保持不变。

```python
SummarizationMiddleware(
  self,
  model: str | BaseChatModel,
  *,
  trigger: ContextSize | TriggerClause | list[ContextSize | TriggerClause] | None = None,
  keep: ContextSize = ('messages', _DEFAULT_MESSAGES_TO_KEEP),
  token_counter: TokenCounter = count_tokens_approximately,
  summary_prompt: str = DEFAULT_SUMMARY_PROMPT,
  trim_tokens_to_summarize: int | None = _DEFAULT_TRIM_TOKEN_LIMIT,
  **deprecated_kwargs: Any = {}
)
```

参数类型解析:

签名中的类型别名都定义在 `langchain/agents/middleware/summarization.py` 中。

1. ContextFraction = tuple[Literal["fraction"], float] 定义了一个别名类型，表示一个固定格式的元组（Tuple）。
    * 元组第一个元素必须是字符串字面量 "fraction"，不能是其他任何字符串。
    * 元组第二个元素必须是一个浮点数（小数）
    * 取值范围是 0 < 值 ≤ 1，表示模型最大输入 token 数（profile 中的 `max_input_tokens`）的比例，例如 `("fraction", 0.8)` 表示 80%。超出范围时，创建中间件就会报 `ValueError`。
2. ContextTokens = tuple[Literal["tokens"], int]：第一个元素固定为 "tokens"，第二个元素是正整数，表示 token 数，例如 `("tokens", 3000)`。
3. ContextMessages = tuple[Literal["messages"], int]：第一个元素固定为 "messages"，第二个元素是正整数，表示消息条数，例如 `("messages", 50)`。
4. ContextSize = ContextFraction | ContextTokens | ContextMessages：`|` 表示“或”，即上面三种元组任选一种。`keep` 的类型就是 `ContextSize`，所以 `keep` 只能写一个元组。
5. TriggerClause：一个 `TypedDict`，即键名和值类型都固定的字典。

    ```python
    class TriggerClause(TypedDict, total=False):
        tokens: int      # token 数达到该值
        messages: int    # 消息条数达到该值
        fraction: float  # token 数达到 max_input_tokens × 该比例
    ```

    * `total=False` 表示三个键都可以省略，但至少要写一个，空字典会报错。
    * 字典里的多个条件要**同时满足**（AND）才触发。例如 `{"tokens": 4000, "messages": 10}` 表示 token 数 ≥ 4000 **并且** 消息数 ≥ 10。
    * 只写一个键时与元组等价：`{"messages": 50}` 和 `("messages", 50)` 效果相同。
6. TokenCounter = Callable[[Iterable[MessageLikeRepresentation]], int]：函数类型，接收一组消息，返回一个整数（token 数）。默认的 `count_tokens_approximately` 按字符数估算；使用默认值时，LangChain 还会自动校准：如果消息中带有模型返回的真实 token 用量（`AIMessage.usage_metadata`），就用它修正估算值。

把这些类型组合起来，`trigger` 的类型 `ContextSize | TriggerClause | list[ContextSize | TriggerClause] | None` 共有四种写法：

| 写法 | 示例 | 何时触发 |
|---|---|---|
| 单个元组 | `("messages", 50)` | 满足这一个条件 |
| 单个字典 | `{"tokens": 4000, "messages": 10}` | 字典里的条件**全部**满足（AND） |
| 列表 | `[("fraction", 0.8), {"tokens": 4000, "messages": 10}]` | 元组和字典可以混写，**任意一项**满足（OR） |
| `None`（默认值） | 不传 `trigger` | 永远不触发 |

后面例子中的 `[("tokens", 100), ("messages", 6), ("fraction", 0.001)]` 就是列表写法，三个条件满足任意一个就会摘要。

其余参数的类型：

* `model: str | BaseChatModel`：模型名称字符串或模型对象，见 2.1.1。
* `summary_prompt: str`：普通字符串。内部用 `summary_prompt.format(messages=...)` 填入待摘要的消息，所以必须包含 `{messages}`；如果提示词里还有别的花括号（例如 JSON 示例），要写成 `{{`、`}}`，否则会报错。另外，摘要写回消息列表时会固定加上英文前缀 `Here is a summary of the conversation to date:`，不受 `summary_prompt` 影响（举例 2 的输出中可以看到）。
* `trim_tokens_to_summarize: int | None`：整数表示只把最近这么多 token 的消息交给摘要模型；传 `None` 表示不裁剪，全部交给摘要模型。
* `**deprecated_kwargs: Any`：接收旧版本的参数名。`max_tokens_before_summary=N` 会转换成 `trigger=("tokens", N)`，`messages_to_keep=N` 会转换成 `keep=("messages", N)`，同时给出弃用警告。副作用是**拼错的参数名也会被它收走，既不报错也不生效**：例如把 `trim_tokens_to_summarize` 写成 `trim_token_to_summarize`，这个设置会被静默忽略。

### 2.1.1 参数说明

1. 参数1：model — 用于摘要的模型

可以是模型名称也可以是模型对象，如果传递的是模型名称，底层会调用 init_chat_model 初始化模型。

2. 参数2：trigger — 摘要触发条件

可以写成单个元组、单个字典或列表，详见上方“参数类型解析”。最常用的是列表写法：每个元素对应一个条件，当 任一条件满足 时，触发摘要。

    * tokens ：token的数量，历史token的累计数量达到该值触发摘要。
    * messages ：历史消息数量，历史消息条数达到该值触发摘要。
    * fraction ：上下文长度比例。历史token的累计数量达到模型的 max_input_tokens*fraction 触发摘要

如果条件包含 fraction ，要求摘要模型（`model` 参数）的 profile 包含 max_input_tokens 。本项目用的 deepseek-v4-pro、deepseek-v4-flash（100 万 token）和 gpt-6-luna（105 万 token）都自带 profile，可以直接使用。deepseek-chat、deepseek-reasoner 这类旧模型名以及 LangChain 不认识的模型 profile 为空，创建中间件时会报 `ValueError`，需要手动传入 profile：例如 DeepSeek-V3.2 的上下文长度为 128K，就传 `profile={"max_input_tokens": 128_000}`（即 2.1.2 中的 `custom_profile`）。

3. 参数3：keep —摘要时保留的原始消息

支持三种条件，但和trigger不同，keep同一时间只接收一种条件。

    * tokens ：摘要时保留的token数量。
    * messages ：摘要时保留的历史消息条数。
    * fraction ：摘要时保留 max_input_tokens*fraction 个token。

4. 参数4：token_counter — 统计token数量的函数

默认使用LangChain提供的 count_tokens_approximately ，一般不用更改。
> 对于纯文本消息，该函数的大致思路是先统计消息的字符数，也就是 len(字符串) ，然后再除以每个token大致的字符数，转换为粗略的token数，再加一些额外开销。作为估算的token数。

5. 参数5：summary_prompt — 用于生成摘要的提示词

该提示词需要包含 {messages} 占位符，使得历史消息列表可以被插入。不指定则使用内置提示词。

6. 参数6：trim_tokens_to_summarize —摘要时历史消息的最大token数

如果历史消息token数大于该值，则会被裁剪。默认为" 4000 "。

如果trigger用token作为度量，调大触发阈值时，当前配置项应相应调整，否则会丢失信息。


### 2.1.2 举例

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]
# 用到 fraction 时，如果模型内部的profile没有指定max_input_tokens,那么必须自定义profile,否则创建SummarizationMiddleware时会报错 ValueError
agent = create_agent(
    model=model,
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=[
            ("tokens", 100),
            ("messages", 6),
            ("fraction", 0.001)
        ],
        keep=("messages", 2)
    )],
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
友好地进行中文问候与自我介绍。

## SUMMARY
用户自称老王。助手自称小王。双方已互相问候并表示很高兴认识对方。

## ARTIFACTS
None

## NEXT STEPS
None
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，开个玩笑啦，没别的意思。咱们才刚认识，后面还有很多话可以聊呢！很高兴认识你，老王。


1. trigger参数定义的触发条件至少有一个满足，触发了摘要。
2. 摘要结果作为HumanMessage，传入消息列表头部
3. keep要求保留两条消息，则最新两条消息原样保留。

### 2.1.3 举例2
测试summary_prompt参数

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]
# 用到 fraction 时，如果模型内部的profile没有指定max_input_tokens,那么必须自定义profile,否则创建SummarizationMiddleware时会报错 ValueError
agent = create_agent(
    model=model,
    middleware=[SummarizationMiddleware(
        model=model,
        trigger=[
            ("tokens", 100),
            ("messages", 6),
            ("fraction", 0.0001)
        ],
        keep=("messages", 2),
        summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
    )],
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

老王向助手打招呼并询问身份。助手自称小王，双方互相问候并表示很高兴认识对方。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

没别的意思，刚才只是想开个玩笑：认识我以后，可能还会有更多“惊喜”呢。要是听起来有点欠揍，那我收回，重新打个招呼：很高兴认识你！


分析:

1. 摘要结果和案例一明显不同，提示词生效了
2. 摘要包含了历史对话记录， {messages} 被替换为历史消息列表。

### 2.1.4 使用注意事项

1. **不传 `trigger` 就永远不会摘要。** `trigger` 的默认值是 `None`，此时中间件什么也不做，也不会报错。只写 `SummarizationMiddleware(model=model)` 等于没加。

2. **写在 `messages` 里的 `SystemMessage` 会被摘要掉。** 回头看前两个例子的输出：开头的 `SystemMessage("你是个非常友好的AI助手")` 不见了。它和普通消息一起被压缩、删除，之后的模型调用再也收不到这条角色设定。角色设定应该用 `create_agent(system_prompt=...)` 传入：`system_prompt` 不存放在消息列表里，每次调用模型时才加到最前面，所以不会被摘要。

3. **摘要会直接改写 state，被摘要的原始消息会被删除。** 中间件先用 `RemoveMessage(id=REMOVE_ALL_MESSAGES)` 清空消息列表，再写入“摘要 + 保留的消息”。配合 checkpointer 进行多轮对话时，后续轮次只能看到摘要；如果前端要展示完整的聊天记录，需要另外保存。

4. **实际保留的消息可能比 `keep` 多。** 为了不把 AIMessage 中的 `tool_calls` 和对应的 `ToolMessage` 拆开，如果切分点落在 `ToolMessage` 上，会往前挪到发起调用的那条 AIMessage。例如 `keep=("messages", 2)`，而最后三条是“一条发起两个工具调用的 AIMessage + 两条 ToolMessage”，这三条会全部保留。

5. **`fraction` 按摘要模型的 `max_input_tokens` 计算。** 读取的是 `SummarizationMiddleware(model=...)` 这个摘要模型的 profile，而不是 Agent 主模型的，两者用不同模型时要留意。本项目用的 `deepseek-v4-pro`、`deepseek-v4-flash`（100 万 token）和 `gpt-6-luna`（105 万 token）都自带 profile，可以直接用 `fraction`；`deepseek-chat`、`deepseek-reasoner` 这类旧模型名以及不认识的模型没有 profile，创建中间件时会直接报 `ValueError`，需要手动传入 `profile={"max_input_tokens": ...}`（见 2.1.2 中注释掉的 `custom_profile`）。

6. **流式输出时，摘要内容也会被推出来。** 用 `stream_mode="messages"` 流式输出时，摘要模型生成的 token 和写回的摘要消息会和主模型的回答一起推出，它们的 `metadata["langgraph_node"]` 是 `"SummarizationMiddleware.before_model"`。前端只想显示 Agent 的回答时，按 `metadata["langgraph_node"] == "model"` 过滤即可。